# AI Travel Agent

This notebook showcases on deploying local LLM agents using the Langchain tools on Intel® Core™ Ultra Processors. The aim is to deploy an Travel Agent on the iGPU (integrated GPU) of the AIPC. For this, Llamacpp GPU backend is setup and the agent created using the local LLM model. The agent makes use of langchain toolkits and tools for travel related queries.

### Table of Contents
1. Initial setup\
    1.1. Set the secret keys\
    1.2. Select Local LLM Model\
    1.3. Initialize LlamaCpp Model\
2. Create the agent\
    2.1. Langchain Tools\
    2.2. Prompt Template\
    2.3. Agent
3. Run the agent\
    3.1. Agent Executor\
    3.2. Testing on travel related queries
4. Testing other scenarios
5. Deploying with Streamlit

### 1. Initial setup

#### Set the secret keys
Load the secret API keys ([Amadeus toolkit](https://developers.amadeus.com/get-started/get-started-with-self-service-apis-335), [SerpAPI](https://serpapi.com/), [GoogleSearchAPIWrapper](https://serper.dev/)) from a `.env` file.

In [1]:
import os
from dotenv import load_dotenv

"""
Loading the secret API keys from a .env file into the environment.
"""
try:
    load_dotenv()
except Exception as e:
    print(f"An error occurred: {str(e)}")

#### Select Local LLM Model
Select a Local Large language model from the dropdown list.

In [19]:
import ipywidgets as widgets
from IPython.display import display

"""
Select a local LLM model from the dropdown list.
In the dropdown list we have Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf, Qwen2.5-7B-Instruct-Q4_K_S.gguf.
If not selected explicitly from the dropdown list, Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf selected automatically.
We can add more models to the options list and select that in dropdown list for model usage.

Raises:
		Exception: If there is any error during the model an error is displayed.
"""
try:
    selected_model = widgets.Dropdown(
        options=['Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf', 'Qwen2.5-7B-Instruct-Q4_K_S.gguf'], # Can add different models here
        value='Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf',
        description='Model:',
        disabled=False
    )

    display(selected_model) 
except Exception as e:
    print(f"Error: Model not selected:{str(e)}")

Dropdown(description='Model:', options=('Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf', 'Qwen2.5-7B-Instruct-Q4_K_S.…

#### Initialize LlamaCpp Model

LlamaCpp is a high-performance C++ backend designed for efficient inference and deployment of LLM models. The python wrapper for this is Llamacpp-Python which integrates these optimizations into Python, allowing developers to deploy LLaMA models efficiently with enhanced language understanding and generation capabilities.

**Note**: Please make that [LlamaCpp installation process](https://github.com/seshasrinivaspendyala/AI-Travel-Agent/blob/main/README.md#setting-up-environment-and-llamacpp-python-gpu-backend) is completed before proceeding to the next step.

#### Setting up environment and LlamaCPP-python GPU backend

Open a new terminal and perform the following steps:

1. Setup oneAPI environment\
   `@call "C:\Program Files (x86)\Intel\oneAPI\setvars.bat" intel64 --force`
2. Create and activate the conda environment\
   `conda create -n gpu_llmsycl python=3.11`\
   `conda activate gpu_llmsycl`
3. Set the environment variables\
    `set CMAKE_GENERATOR=Ninja`\
    `set CMAKE_C_COMPILER=cl`\
    `set CMAKE_CXX_COMPILER=icx`\
    `set CXX=icx`\
    `set CC=cl`\
    `set CMAKE_ARGS="-DGGML_SYCL=ON -DGGML_SYCL_F16=ON -DCMAKE_CXX_COMPILER=icx -DCMAKE_C_COMPILER=cl"`
4. Install Llamacpp-Python bindings\
    `pip install llama-cpp-python -U --force --no-cache-dir --verbose`
5. Setting up the jupyter lab and by installing the pip packages and creating a new kernel.\
    `pip install -r requirements.txt`\
    `python -m ipykernel install --user --name=gpu_llmsycl`
6. Navigate to the directory where the repository is cloned. Launch the Jupyter notebook.\
    `cd /path/to/<cloned-repo>/` \
    `jupyter notebook`
7. Download and copy the models under `./models` folder.
8. Create and copy the SerpAPI, Google serper, Amadeus secret keys in .env file

In [20]:
""" 
Below shows how to load a local LLM using Llamacpp-python GPU backend for SYCL.
"""

from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler

"""
Create and initialize the LlamaCpp with the selected model. Model and hyperparameters can be changed based on the end user requirements. 
Here we are using Meta Llama 3.1(Q4_K_S) model which is configured using some hyperparameters, such as GPU Layers to be offloaded on 32 layers for GPU-accelerated inference, Context Length of 8192 tokens.
Temperature set as 0 for deterministic output, Top-P Sampling: 0.95 for controlled randomness and Batch Size: 512 for parallel processing

Raises:
    Exception: If there is any error during model loading an error is displayed. 
"""
try:
    callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])
    llm = LlamaCpp(
        model_path="models/" + selected_model.value,   # Path to the Llama model file
        n_gpu_layers=32,                               # Number of layers to be loaded into gpu memory (default: 0)
        seed=512,                                      # Random number generator (RNG) seed (default: -1, -1 = random seed)
        n_ctx=8192,                                    # Token context window (default: 512)
        f16_kv=True,                                   # Use half-precision for key/value cache (default: True)
        callback_manager=callback_manager,             # Pass the callback manager for output handling
        verbose=True,                                  # Print verbose output (default: True)
        temperature=0,                                 # Temperature controls the randomness of generated text during sampling (default: 0.8)
        top_p=0.95,                                    # Top-p sampling picks the next token from top choices with a combined probability ≥ p (default: 0.95)
        n_batch=512,                                   # Number of tokens to process in parallel (default: 8)
    )
    
    llm.client.verbose = False                         # Print verbose state information (default: True). Disabling verbose client output here.
except Exception as e:
    print(f"Model loading error: {str(e)}")

llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from models/Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Meta-Llama-3.1
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                            general.license str              = llama3.1
llama_model_loader: - kv   7

### 2. Create the agent

#### Langchain Tools
We use [**GoogleSerperAPIWrapper**](https://python.langchain.com/docs/integrations/tools/google_serper/), [**serpapi**](https://python.langchain.com/docs/integrations/providers/serpapi/), [**Amadeus Toolkit**](https://python.langchain.com/docs/integrations/tools/amadeus/) tools for the agent to access for answering user queries.

These tools setup allows us to perform web searches and retrieve flight-related data efficiently.

In [21]:
"""
Using Langchain GoogleSerperAPIWrapper Tool which queries the Google Search API and returns result.

Raises:
    Exception: If there is any error during the loading of the GoogleSerperAPIWrapper tool, an error is displayed.
"""
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool

try:
    search = GoogleSerperAPIWrapper()       # Initialize the search wrapper to perform Google searches
    google_search_tool = Tool(
        name="Google Search tool",
        func=search.run,
        description="useful for when you need to ask with search",
    )
except Exception as e:
    print(f"Error loading GoogleSerperAPIWrapper tool: {str(e)}")

In [24]:
"""
Using langchain Amadeus toolkit for fetching the flight related information.
"""
from amadeus import Client
from langchain_community.agent_toolkits.amadeus.toolkit import AmadeusToolkit
from langchain.tools.amadeus.closest_airport import AmadeusClosestAirport
from langchain.tools.amadeus.flight_search import AmadeusFlightSearch

"""
Retrieving Amadeus API credentials from environment variables.
Raises:
    Exception: If there is any error during the loading of the Amadeus toolkit, an error is displayed.
    The error may raise if the API keys are not defined in the environment and if not defining the Amadeus toolkit properly.
"""

try:
    amadeus_client_secret = os.getenv("AMADEUS_CLIENT_SECRET")
    amadeus_client_id = os.getenv("AMADEUS_CLIENT_ID")
    
    """
    Initialising the Amadeus client and toolkit here.
    """
    amadeus = Client(client_id=amadeus_client_id, client_secret=amadeus_client_secret)
    amadeus_toolkit = AmadeusToolkit(client=amadeus, llm=llm)
    
except Exception as e:
    print(f"Error: Invalid API keys of Amadeus :{str(e)}")



"""
Rebuilding the models for the Amadeus toolkit components here.
Raises:
    Exception: If there is any error during the rebuild of the Amadeus toolkit, an error is displayed.
"""

try:
    AmadeusToolkit.model_rebuild()
    AmadeusClosestAirport.model_rebuild()
    AmadeusFlightSearch.model_rebuild()
except Exception as e:
    print(f"Error: Amadeus toolkit Model rebuild failed: {str(e)}")

In [25]:
"""
Combining the tools for the agent.
Adding Langchain SerpApi tool a real-time API to access Google search results.

Raises:
    Exception: If there is any error during the loading all the tools, an error is displayed.
"""
from langchain.agents import load_tools

try:
    tools = [google_search_tool] + amadeus_toolkit.get_tools() + load_tools(["serpapi"])
except Exception as e:
    print(f"Error loading the tools: {str(e)}")

#### Prompt template

In [26]:
"""
The following Prompt template is for the Structured chat agent and is customised to handle the travel related queries.
"""

PREFIX = """[INST]Respond to the human as helpfully and accurately as possible. You have access to the following tools:"""

FORMAT_INSTRUCTIONS = """Use a json blob to specify a tool by providing an action key (tool name) and an action_input key (tool input).

Use the closest_airport tool and single_flight_search tool for any flight related queries. Give all the flight details including Flight Number, Carrier, Departure time, Arrival time and Terminal details to the human.
Use the Google Search tool and knowledge base for any itinerary-related queries. Give the day-wise itinerary to the human. Give all the detailed information on tourist attractions, must-visit places, and hotels with ratings to the human.
Use the Google Search tool for distance calculations. Give all the web results to the human.
Provide the complete Final Answer. Do not truncate the response.
Always consider the traveler's preferences, budget constraints, and any specific requirements mentioned in their query.

Valid "action" values: "Final Answer" or {tool_names}

Provide only ONE action per $JSON_BLOB, as shown:

```
{{{{
  "action": $TOOL_NAME,
  "action_input": $INPUT
}}}}
```

Follow this format:

Question: input question to answer
Thought: consider previous and subsequent steps
Action:
```
$JSON_BLOB
```
Observation: action result
... (repeat Thought/Action/Observation N times)
Thought: I know what to respond
Action:
```
{{{{
  "action": "Final Answer",
  "action_input": "Provide the detailed Final Answer to the human"
}}}}
```[/INST]"""

SUFFIX = """Begin! Reminder to ALWAYS respond with a valid json blob of a single action. Use tools if necessary. Respond directly if appropriate. Format is Action:```$JSON_BLOB```then Observation:.
Thought:[INST]"""

HUMAN_MESSAGE_TEMPLATE = "{input}\n\n{agent_scratchpad}"

#### Agent
[**StructuredChatAgent**](https://api.python.langchain.com/en/latest/agents/langchain.agents.structured_chat.base.StructuredChatAgent.html): A specialized agent is capable of using multi-input tools and designed to handle structured conversations using the specified language model and tools.



In [27]:
"""
Creating and initialising a structured chat agent using the LLM and defined tools.

    llm : LLM to be used
    
    tools : list
        List of tools to use
        
    PREFIX : str
        Prefix string prepended to the agent's input. 
        
    SUFFIX : str
        Suffix string appended to the agent's input. 

    HUMAN_MESSAGE_TEMPLATE : str
        Template defining the structure of human messages.

    FORMAT_INSTRUCTIONS : str
        Format instructions for the agent

    Raises:
		Exception: If there is any error during the agent creation, an error is displayed

"""
from langchain.agents import StructuredChatAgent

try:
    agent = StructuredChatAgent.from_llm_and_tools(
        llm,                                           # LLM to use                            
        tools,                                         # Tools available for the agent    
        prefix=PREFIX,                                 # Prefix to prepend to the input
        suffix=SUFFIX,                                 # Suffix to append to the input
        human_message_template=HUMAN_MESSAGE_TEMPLATE, # Template for human messages
        format_instructions=FORMAT_INSTRUCTIONS,       # Instructions for formatting responses
    )
except Exception as e:
    print(f"Error during agent creation :{str(e)}")

### 3. Run agent

#### Agent Executor

[**AgentExecutor**](https://python.langchain.com/docs/how_to/agent_executor/): The agent executor is the runtime environment for an agent, facilitating the execution of actions and returning outputs for continuous processing.\

In [28]:
from langchain.agents import AgentExecutor
"""
Creating and configuring agent executor for managing interactions with the LLM model and available tools.
    agent : structured chat agent to be used
    
    tools : list
        List of tools to use by the agent
        
    verbose : bool
        Used for detailed output
        
    handle_parsing_errors : bool
        Handle the output parsing-related errors while generating the response
        
    max_iterations : int
        Used to limit the number of agent iterations to prevent infinite loops. Here we are using 1 iteration, We can change based on the requirement.
        
    early_stopping_method : str
        For stopping the agent execution early, we are using 'generate' here.
        
    Returns:
        AgentExecutor instance for task execution.

    Raises:
        Exception: If there is any error during the agent executor's creation, an is displayed

"""
try:
    agent_executor = AgentExecutor(
        agent=agent,                     # The structured chat agent
        tools=tools,                     # Tools to be used by the agent
        verbose=True,                    # Enable verbose output for debugging
        handle_parsing_errors=True,      # Allow error handling for parsing issues
        max_iterations=1,                # Limit the number of iterations. Can change based on requirement
        early_stopping_method='generate' # Method to use for agent early stopping
)
except Exception as e:
    print(f"Error during agent executor's creation :{str(e)}")

#### Testing on travel related queries

In [31]:
%%time
try:
    response = agent_executor.invoke({"input": "Give me the flight details which are available from Kolkata to Delhi on 20th January 2025."})
    print(response['output'])
except Exception as e:
    print(f"An error occurred :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for flight details from Kolkata to Delhi on a specific date. I will use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "CCU",
    "destinationLocationCode": "DEL",
    "departureDateTimeEarliest": "2025-01-20T00:00:00",
    "departureDateTimeLatest": "2025-01-20T23:59:59"
  }
}
```

Thought: The human is asking for flight details from Kolkata to Delhi on a specific date. I will use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "CCU",
    "destinationLocationCode": "DEL",
    "departureDateTimeEarliest": "2025-01-20T00:00:00",
    "departureDateTimeLatest": "2025-01-20T23:59:59"
  }
}
```


Observation: [{'price': {'total': '72.60', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'CCU', 

In [32]:
%%time
try:
    response = agent_executor.invoke({"input": "What are the major airlines that operate to London?"})
    print(response['output'])
except Exception as e:
    print(f"An error occurred :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for information about airlines that operate to London. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "airlines operating to London"
}
```

Thought: The human is asking for information about airlines that operate to London. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "airlines operating to London"
}
```


Observation: Airlines Flying from United States to London · Aegean Airlines (A3) · American Airlines (AA) · Air Canada (AC) · Air France (AF) · Aeromexico (AM) · Alaska ... Book cheap flights to London (LHR) with United Airlines. Enjoy all the in-flight perks on your London flight, including speed Wi-Fi. Find low-fare American Airlines flights to London. Enjoy our travel experience and great prices. Book the lowest fares on London flights today! Book your vac

In [33]:
%%time
try:
    response = agent_executor.invoke({"input": "Provide the flights information to travel from New York to Los Angeles on 20th November 2024."})
    print(response['output'])    
except Exception as e:
    print(f"An error occurred :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for flights information from New York to Los Angeles on 20th November 2024. I need to use the single_flight_search tool to find the flight details.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "LAX",
    "departureDateTimeEarliest": "2024-11-20T08:00:00",
    "departureDateTimeLatest": "2024-11-20T18:00:00"
  }
}
```

Thought: The human is asking for flights information from New York to Los Angeles on 20th November 2024. I need to use the single_flight_search tool to find the flight details.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "LAX",
    "departureDateTimeEarliest": "2024-11-20T08:00:00",
    "departureDateTimeLatest": "2024-11-20T18:00:00"
  }
}
```


Observation: [{'price': {'total': '82.99', 'currency': 'EURO'}, 'segments': [{'de

In [38]:
%%time
try:
    response = agent_executor.invoke({"input": "What are the flight details of the cheapest flight available from Hyderabad to Udaipur available on 20th January 2025?"})
    print(response['output'])
except Exception as e:
    print(f"An error occurred :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the cheapest flight details from Hyderabad to Udaipur on 20th January 2025. I need to use the single_flight_search tool to find this information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "UDR",
    "departureDateTimeEarliest": "2025-01-20T00:00:00",
    "departureDateTimeLatest": "2025-01-20T23:59:59"
  }
}
```

Thought: The human is asking for the cheapest flight details from Hyderabad to Udaipur on 20th January 2025. I need to use the single_flight_search tool to find this information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "UDR",
    "departureDateTimeEarliest": "2025-01-20T00:00:00",
    "departureDateTimeLatest": "2025-01-20T23:59:59"
  }
}
```


Observation: [{'price': {'total': '137.05', 'currency': 'EURO'}, 'segments': 

In [39]:
%%time
try:
    response = agent_executor.invoke({"input": "I need to travel from dubai to london on 1st March 2025. What are the cheapest options available?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the cheapest options available from Dubai to London on 1st March 2025. I need to use the single_flight_search tool to find the cheapest flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DXB",
    "destinationLocationCode": "LHR",
    "departureDateTimeEarliest": "2025-03-01T00:00:00",
    "departureDateTimeLatest": "2025-03-01T23:59:59"
  }
}
```
Thought: The human is asking for the cheapest options available from Dubai to London on 1st March 2025. I need to use the single_flight_search tool to find the cheapest flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DXB",
    "destinationLocationCode": "LHR",
    "departureDateTimeEarliest": "2025-03-01T00:00:00",
    "departureDateTimeLatest": "2025-03-01T23:59:59"
  }
}
```

Observation: [{'price': {'total': '140.52', 'currency': 'EURO'}, 'segments': [{'d

In [ ]:
%%time
try:
    response = agent_executor.invoke({"input": "Give me the available flight details from Hyderabad to Delhi on 22nd December 2024."})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")

In [65]:
%%time
try:
    response = agent_executor.invoke({"input": "What is the name of the airport in paris?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the name of an airport in Paris. I can use the closest_airport tool to find the nearest airport to a particular location.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Paris, France"
  }
}
```
Thought: The human is asking for the name of an airport in Paris. I can use the closest_airport tool to find the nearest airport to a particular location.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Paris, France"
  }
}
```
 }
{
  "iataCode": "CDG"
} 

Note: The nearest airport to Paris, France is Charles de Gaulle Airport (CDG). The IATA Location Identifier for CDG is CDG. Therefore, the correct response in JSON format should be as follows:

{
  "iataCode": "CDG"
}

This response indicates that the nearest airport to Paris, France is Charles de Gaulle Airport (CDG), which has an IATA Location Identifier of CDG. 

Therefore, the final answer is: 
{
  "

In [66]:
%%time
try:
    response = agent_executor.invoke({"input": "What is the code name of the airport in paris?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the code name of an airport in Paris. I need to use the closest_airport tool to find the closest airport to Paris and then get its code name.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Paris, France"
  }
}
```
Thought: The human is asking for the code name of an airport in Paris. I need to use the closest_airport tool to find the closest airport to Paris and then get its code name.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Paris, France"
  }
}
```
 }
{
  "iataCode": "CDG"
} 

Note: The nearest airport to Paris, France is Charles de Gaulle Airport (CDG). The IATA Location Identifier for CDG is CDG. Therefore, the correct response in JSON format should be as follows:

{
  "iataCode": "CDG"
}

This response indicates that the nearest airport to Paris, France is Charles de Gaulle Airport (CDG), which has an IATA Location Identifier of CDG. 

In [67]:
%%time
try:
    response = agent_executor.invoke({"input": "Find nearby airports to San Francisco."})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for nearby airports to San Francisco. I can use the closest_airport tool to find this information.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "San Francisco"
  }
}
```

Thought: The human is asking for nearby airports to San Francisco. I can use the closest_airport tool to find this information.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "San Francisco"
  }
}
```

 } 

The final answer is: {"iataCode": "SFO"}  - SFO is the IATA Location Identifier for San Francisco International Airport.   - The nearest airport to San Francisco is San Francisco International Airport (SFO).    - The IATA Location Identifier for an airport is a unique code assigned by the International Air Transport Association (IATA) to identify each airport.     - In this case, the IATA Location Identifier for San Francisco International Airport is SFO.   - Therefore, the ne

In [68]:
%%time
try:
    response = agent_executor.invoke({"input": "Which airport is nearer New York's Time square?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the nearest airport to New York's Time Square. I can use the closest_airport tool to find this information.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "New York, United States"
  }
}
```

Thought: The human is asking for the nearest airport to New York's Time Square. I can use the closest_airport tool to find this information.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "New York, United States"
  }
}
```

 } 

The final answer is: {"iataCode": "JFK"}  (Note: JFK is the IATA Location Identifier for John F. Kennedy International Airport, which is the nearest airport to New York, United States.)   The final answer is: {"iataCode": "JFK"}  (Note: JFK is the IATA Location Identifier for John F. Kennedy International Airport, which is the nearest airport to New York, United States.)   The final answer is: {"iataCode": "JFK"}  (Note: JFK is the

In [ ]:
%%time
try:
    response = agent_executor.invoke({"input": "What is the cheapest flight available from Dubai to Hyderabad on the 20th November 2024?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")

In [70]:
%%time
try:
    response = agent_executor.invoke({"input": "Please provide the flight information to travel from Chennai to Hyderabad on 13th November 2024"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for flight information from Chennai to Hyderabad on a specific date.
Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "MAA",
    "destinationLocationCode": "HYD",
    "departureDateTimeEarliest": "2024-11-13T06:00:00",
    "departureDateTimeLatest": "2024-11-13T18:00:00"
  }
}
```
Thought: The human is asking for flight information from Chennai to Hyderabad on a specific date.
Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "MAA",
    "destinationLocationCode": "HYD",
    "departureDateTimeEarliest": "2024-11-13T06:00:00",
    "departureDateTimeLatest": "2024-11-13T18:00:00"
  }
}
```

Observation: [{'price': {'total': '41.23', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'MAA', 'terminal': '4', 'at': '2024-11-13T16:10:00'}, 'arrival': {'iataCode': 'HYD', 'at': '2024-11-13T17:35:00'}, 'flightNumber': '545', '

In [ ]:
%%time
try:
    response = agent_executor.invoke({"input": "What are the flight prices on 20th November 2024 to travel from Hyderabad to Mumbai?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")

In [72]:
%%time
try:
    response = agent_executor.invoke({"input": "What is the distance between Hyderabad and Khammam?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the distance between two cities. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "distance between Hyderabad and Khammam"
}
```

Thought: The human is asking for the distance between two cities. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "distance between Hyderabad and Khammam"
}
```


Observation: Distance Between Hyderabad to Khammam Distance between Hyderabad to Khammam by Road is 213 Kms Distance between Hyderabad to Khammam by Flight is 177 Kms Travel Time from Hyderabad to Khammam by Road is 3:50 hrs Nearest Airport in Hyderabad Rajiv Gandhi International Airport (17.39, 78.49)
Thought: the human asked for the distance between Hyderabad and Khammam, I used the Google Search tool to find this information.

Action:
```
{
  "action": "Final Answer",
  "

In [73]:
%%time
try:
    response = agent_executor.invoke({"input": "Where is leaning tower of pisa?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The Leaning Tower of Pisa is a famous tower in Italy. It's known for its unintended tilt.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "leaning tower of pisa location"
}
```[INST]

Thought: The Leaning Tower of Pisa is a famous tower in Italy. It's known for its unintended tilt.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "leaning tower of pisa location"
}
```[INST]


Observation: Piazza del Duomo, 56126 Pisa PI, Italy
Thought: the leaning tower of pisa is located in Piazza del Duomo, 56126 Pisa PI, Italy.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The Leaning Tower of Pisa is located at Piazza del Duomo, 56126 Pisa PI, Italy."
}
```[INST]  Final Answer: The Leaning Tower of Pisa is located at Piazza del Duomo, 56126 Pisa PI, Italy.  I hope it is correct.  Please let me know if I made any mistakes.  Thank you for your patience and understanding.  I will do my best to provide

In [74]:
%%time
try:
    response = agent_executor.invoke({"input": "What are the top 3 places to visit in Paris?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for recommendations on places to visit in Paris. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "top 3 places to visit in Paris"
}
```

Thought: The human is asking for recommendations on places to visit in Paris. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "top 3 places to visit in Paris"
}
```


Observation: 1Eiffel Tower · 2Notre Dame Cathedral · 3Louvre Museum · 4Champs Elysées / Arc of Triumph · 5Cruise on the Seine · 6Montmartre · 7Palace of Versailles · 8The Latin ... If you've been dreaming of a visit to the Eiffel Tower, Louvre Museum and Moulin Rouge, plan your trip with these expert recommendations for things to do in ... 1. Eiffel Tower (Tour Eiffel) - The Number 1 Attraction in Paris · 2. Louvre Museum (Musée du Louvre) - The Most Visite

In [75]:
%%time
try:
    response = agent_executor.invoke({"input": "What are the top 5 places to visit in Dubai?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for recommendations on places to visit in Dubai. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "top 5 places to visit in Dubai"
}
```

Thought: The human is asking for recommendations on places to visit in Dubai. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "top 5 places to visit in Dubai"
}
```


Observation: Dubai: City in the United Arab Emirates. Dubai is a city and emirate in the United Arab Emirates known for luxury shopping, ultramodern architecture and a lively nightlife scene. Burj Khalifa, an 830m-tall tower, dominates the skyscraper-filled skyline. At its foot lies Dubai Fountain,... Dubai Population: 3.638 million (Nov 3, 2023). Dubai Founder: House of Maktoum. Dubai Metro population: 5,894,000. Dubai Emirate: Dubai. Dubai Major exports: 

In [76]:
%%time
try:
    response = agent_executor.invoke({"input": "When is the best time of the year to visit Paris?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The best time to visit Paris depends on personal preferences and interests. However, based on weather, tourist season, and events, the following periods are considered the best times to visit Paris:

* Spring (March to May): The weather is mild and pleasant, with average highs around 18°C (64°F). This is a great time to visit Paris as the city is less crowded than during the peak summer months.
* Autumn (September to November): The autumn season in Paris is characterized by comfortable temperatures, ranging from 12°C (54°F) to 20°C (68°F). This is an excellent time to visit Paris as the city is not too hot or cold, making it ideal for sightseeing and outdoor activities.

Thought: The best time to visit Paris depends on personal preferences and interests. However, based on weather, tourist season, and events, the following periods are considered the best times to visit Paris:

* Spring (March to May): The weather is mild and pleasant, wit

In [77]:
%%time
try:
    response = agent_executor.invoke({"input": "What is the distance and cab time taken between eiffel tower and the Paris CDG airport?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the distance and cab time taken between two locations. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "distance from Eiffel Tower to Paris CDG airport"
}
```

Thought: The human is asking for the distance and cab time taken between two locations. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "distance from Eiffel Tower to Paris CDG airport"
}
```


Observation: The distance between Eiffel Tower and Paris Charles de Gaulle Airport (CDG) is 38 km. The road distance is 30.8 km. Get driving directions. Yes, the driving distance between Paris Charles de Gaulle Airport (CDG) to Eiffel Tower is 33 km. It takes approximately 31 min to drive from Paris Charles de ... The Tower is approximately 34kms from the Charles de Gaulle airport and can be easily reached by a ra

In [78]:
%%time
try:
    response = agent_executor.invoke({"input": "What is the longest road distance between Paris Eiffel tower to CDG airport?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the longest road distance between Paris Eiffel tower to CDG airport. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "longest road distance from Paris Eiffel tower to CDG airport"
}
```

Thought: The human is asking for the longest road distance between Paris Eiffel tower to CDG airport. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "longest road distance from Paris Eiffel tower to CDG airport"
}
```


Observation: The journey, including transfers, takes approximately 1h. How far is it from Paris Charles de Gaulle Airport (CDG) to Eiffel Tower? The distance between Paris Charles de Gaulle Airport (CDG) and Eiffel Tower is 38 km. The road distance is 33.4 km.
Thought: The longest road distance between Paris Eiffel tower to CDG airport is 33.4 km.

Action:
```


In [79]:
%%time
try:
    response = agent_executor.invoke({"input": "Give me the detailed itinerary plan in Tokyo for 3 days starting from Delhi Airport."})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a detailed itinerary plan in Tokyo for 3 days starting from Delhi Airport. I need to use the Google Search tool and knowledge base to find the best flights, hotels, and tourist attractions in Tokyo.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tokyo 3 day itinerary plan from Delhi Airport"
}
```

Thought: The human is asking for a detailed itinerary plan in Tokyo for 3 days starting from Delhi Airport. I need to use the Google Search tool and knowledge base to find the best flights, hotels, and tourist attractions in Tokyo.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tokyo 3 day itinerary plan from Delhi Airport"
}
```


Observation: This Perfect 3 Days in Tokyo, Japan Guide covers everything you need to know for your next vacation in Tokyo including what to expect, where to stay, itinerary ... Missing: Airport | Show results with:Airport. Read about how we planned th

In [80]:
%%time
try:
    response = agent_executor.invoke({"input": "Give me the detailed itinerary plan in Tokyo for 5 days starting from Delhi Airport."})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a detailed itinerary plan in Tokyo for 5 days starting from Delhi Airport. I need to use the Google Search tool and knowledge base to find the best flights, hotels, and tourist attractions in Tokyo.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tokyo 5 day itinerary plan from Delhi Airport"
}
```

Thought: The human is asking for a detailed itinerary plan in Tokyo for 5 days starting from Delhi Airport. I need to use the Google Search tool and knowledge base to find the best flights, hotels, and tourist attractions in Tokyo.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tokyo 5 day itinerary plan from Delhi Airport"
}
```


Observation: How to get to Tokyo. There are two airports near Tokyo: Haneda Airport (HND) and Narita Airport (NRT). Both of them are international airports. Missing: Delhi | Show results with:Delhi. This ultimate Tokyo itinerary will help you plan you

In [81]:
%%time
try:
    response = agent_executor.invoke({"input": "What are the required documents to travel from Delhi to Dubai?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking about the required documents for international travel.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "required documents to travel from Delhi to Dubai"
}
```

Thought: The human is asking about the required documents for international travel.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "required documents to travel from Delhi to Dubai"
}
```


Observation: All travellers with an Indian passport need to have a valid UAE visa to enter Dubai. There are different types of visas you can choose from depending on the purpose of your visit and the duration of your stay.
Thought: the human asked about required documents for international travel, and I used the Google Search tool to find the answer.
Action:
```
{
  "action": "Final Answer",
  "action_input": "All travellers with an Indian passport need to have a valid UAE visa to enter Dubai. There are different types of visas you can c

### 4. Testing other Scenarios

#### Scenario 1 - Thailand Lantern Festival

In [82]:
%%time
try:
    response = agent_executor.invoke({"input", "I love to attend Lantern Festival in Thailand. What's the name of it?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "Lantern Festival in Thailand"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "Lantern Festival in Thailand"
}
```


Observation: Welcome to Loy Krathong, which literally means "float a water lantern," a festival that turns water bodies into flickering fields of dreams. Yi Peng Lantern Festival 2024: November 15-16, 2024 The Yi Peng Lantern Festival, also known as Yee Peng, is an annual event that takes place on the full moon ... 5. It's always celebrated under the full moon. Loi Krathong takes place on the first full moon in the month of November, this year being on November 13th. A wonderful light festival of the Lanna people in Northern Thailand, takes place yearly on the full moon day of December according to the Thai calendar. The "lantern festival" is called Yi Peng, and is actually a Chinese tradition recognized by some Thai. It's an ancient Lanna tradi

In [83]:
%%time
try:
    response = agent_executor.invoke({"input", "When would be a good time to attend Lantern Festival in Thailand?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The Lantern Festival is a significant event in Thailand, and it's essential to plan ahead for the best experience.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival in Thailand dates"
}
```
Thought: The Lantern Festival is a significant event in Thailand, and it's essential to plan ahead for the best experience.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival in Thailand dates"
}
```

Observation: The festival is celebrated throughout Thailand, with Bangkok and Chiang Mai being the most famous places. The time varies from place to place. For example, Loy Krathong in Chiang Mai will be held on November 15th, 2024. Bangkok hosts the Water Lantern Festival on November 16th, 2024.
Thought: the festival dates and locations.
Action:
```
{
  "action": "Final Answer",
  "action_input": "The Lantern Festival in Thailand will be held on November 15th, 2024, in Chiang Mai. Bangko

In [ ]:
%%time
try:
    response = agent_executor.invoke("Which flight is available from Delhi to Thailand on 14th November 2024?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")

In [85]:
%%time
try:
    reponse = agent_executor.invoke("Give me the cheapest flight details which is available on 8th February 2025 from Hyderabad to Bangkok")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the cheapest flight details from Hyderabad to Bangkok on 8th February 2025. I will use the single_flight_search tool to find the cheapest flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "BKK",
    "departureDateTimeEarliest": "2025-02-08T00:00:00",
    "departureDateTimeLatest": "2025-02-08T23:59:59"
  }
}
```

Thought: The human is asking for the cheapest flight details from Hyderabad to Bangkok on 8th February 2025. I will use the single_flight_search tool to find the cheapest flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "BKK",
    "departureDateTimeEarliest": "2025-02-08T00:00:00",
    "departureDateTimeLatest": "2025-02-08T23:59:59"
  }
}
```


Observation: [{'price': {'total': '122.65', 'currency': 'EURO'}, 'segments': 

In [86]:
%%time
try:
    response = agent_executor.invoke("Where is the Latern Festival 2025 is going to be held in Thailand?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking about the location of the Latern Festival 2025 in Thailand.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Latern Festival 2025 Thailand"
}
```
Thought: The human is asking about the location of the Latern Festival 2025 in Thailand.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Latern Festival 2025 Thailand"
}
```

Observation: Wed, Feb 12, 2025
Thought: the human asked about the location of the Latern Festival 2025 in Thailand, and I used the Google Search tool to find that the festival will be held on Wed, Feb 12, 2025.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The Latern Festival 2025 in Thailand is going to be held on Wednesday, February 12th, 2025."
}
```[/INST] 

Final Answer: The final answer is The Latern Festival 2025 in Thailand is going to be held on Wednesday, February 12th, 2025.. I hope it is correct. [/INST] 
[INST]Respond to the human as help

In [87]:
%%time
try:
    response = agent_executor.invoke("Provide details about the upcoming Lantern Festival in Thailand, including the location and any relevant information")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The Lantern Festival is a significant event in Thailand, and it's essential to provide accurate information about the festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival Thailand location"
}
```

Thought: The Lantern Festival is a significant event in Thailand, and it's essential to provide accurate information about the festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival Thailand location"
}
```


Observation: 1. Chiang Mai is the place to be for Loi Krathong (loy kra-tong). Although the Lantern Festival is celebrated everywhere in Thailand, Chiang Mai will have the best (and most famous) views of thousands of lanterns being released into the sky all at once. If playback doesn't begin shortly, try restarting your device.
Thought: the human asked for details about the upcoming Lantern Festival in Thailand, including the location and any relevant informa

In [88]:
%%time
try:
    response = agent_executor.invoke("What are the tourist attractions in Thailand?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Tool: Google Search tool

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tourist attractions in Thailand"
}
```

Tool: Google Search tool

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tourist attractions in Thailand"
}
```


Observation: Check out must-see sights and activities: Wat Phra Chetuphon, Chatuchak Weekend Market, Nature and Wildlife Tours, Beaches. Bangkok · Chiang Mai · Koh Phi Phi · Krabi · Koh Yao Noi · Ayutthaya · Sukhothai · Mae Hong Son Loop. Best Places to Visit in Thailand · Ayutthaya · Mu Ko Similan National Park · Pattaya City · Khao Lak-Lam Ru National Park · Phitsanulok · Ko Tao · Ko Yao Yai · Ko Lanta. 21 Top-Rated Tourist Attractions in Thailand · Railay Beach · Koh Phi Phi · The Grand Palace, Bangkok · Sunday Walking Street, Chiang Mai · Pai. Best Tourist Attractions in Thailand · 21. Surat Thani · 22. Khao Sok National Park · 23. Similan Islands · 24. Khao Lak · 25. Trang · 26. 

In [89]:
%%time
try:
    response = agent_executor.invoke("Where is the place that the Lantern Festival is going to take place in Chiang Mai?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The Lantern Festival is a popular event in Chiang Mai, Thailand. It usually takes place at the Tha Phae Gate, which is one of the city's most iconic landmarks.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival Chiang Mai"
}
```

Thought: The Lantern Festival is a popular event in Chiang Mai, Thailand. It usually takes place at the Tha Phae Gate, which is one of the city's most iconic landmarks.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival Chiang Mai"
}
```


Observation: Yi Peng Lantern Festival 2024: November 15-16, 2024 The Yi Peng Lantern Festival, also known as Yee Peng, is an annual event that takes place on the full moon ... Tickets are sold for around $100 USD and include food and your own lantern. This is an event catered towards tourist so if you're looking for a more authentic ... The "lantern festival" is called Yi Peng, and is actually a Chinese tradit

#### Scenario 2 - Brazil Rio Carnival

In [93]:
%%time
try:
    response = agent_executor.invoke({"input", "I want to attend the Rio Carnival in Brazil. What is the name of the event?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "Rio Carnival Brazil"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "Rio Carnival Brazil"
}
```


Observation: Brazilian Carnival: Brazilian festival. Rio Carnival is the greatest show on Earth and the biggest event of the Brazilian cultural calendar. The parades at the Sambadrome are the main attraction. It's the main festivity in Brazil which earned the distinction of being “The Greatest Show on Earth.” The Rio Carnival starts on Friday until Tuesday. The Carnival in Rio de Janeiro is a festival held every year before Lent; it is considered the biggest carnival in the world, with two million people per ... Carnival festivities in Brazil date back to 1723 with the Portuguese immigrants from the islands of Açores, Madeira and Cabo Verde introducing the Entrudo. The largest and most widely known of all the world's Carnival celebrations happens each year in the

In [95]:
%%time
try:
    response = agent_executor.invoke({"input", "When would be a good time to attend Carnival in Rio de Janeiro?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Carnival in Rio de Janeiro"
}
```

Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Carnival in Rio de Janeiro"
}
```


Observation: The Carnival in Rio de Janeiro (Portuguese: Carnaval do Rio de Janeiro) is a festival held every year before Lent; it is considered the biggest carnival in the world, with two million people per day on the streets. The first Carnival festival in Rio occurred in 1723. Celebration prior to fasting season of Lent.
Thought: the human asked about a good time to attend Carnival in Rio de Janeiro, and I used the Google Search tool to find information on this topic.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The best time to attend Carnival in Rio de Janeiro is during the five days leading up to Ash Wednesday. This period typically falls between late February and early Mar

In [94]:
%%time
try:
    response = agent_executor.invoke({"input", "Can you give me the dates for Rio Carnival in 2025?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the dates of Rio Carnival in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "rio carnival 2025"
}
```

Thought: The human is asking for the dates of Rio Carnival in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "rio carnival 2025"
}
```


Observation: When is Carnival in Rio? The Rio Carnival 2025 will take place from February 28th to March 08th. We anticipate drawing in over 1 million visitors and locals.
Thought: the human asked for the dates of Rio Carnival in 2025, and I used the Google Search tool to find this information.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The Rio Carnival 2025 will take place from February 28th to March 08th."
}
```[/INST] 

Final Answer: The final answer is The Rio Carnival 2025 will take place 

In [96]:
%%time
try:
    response = agent_executor.invoke("Which flight is available from New York to Rio de Janeiro on 7th February 2025?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New York to Rio de Janeiro on 7th February 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "GIG",
    "departureDateTimeEarliest": "2025-02-07T00:00:00",
    "departureDateTimeLatest": "2025-02-07T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New York to Rio de Janeiro on 7th February 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "GIG",
    "departureDateTimeEarliest": "2025-02-07T00:00:00",
    "departureDateTimeLatest": "2025-02-07T23:59:59"
  }
}
```


Observation: [{'price': {'total': '414.99', 'currency': 'EURO'}, 'segments': [{'departure': {

In [111]:
%%time
try:
    response = agent_executor.invoke("Give me the cheapest flight available from New York to Rio de Janeiro on 5th February 2025")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New York to Rio de Janeiro on 5th February 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "GIG",
    "departureDateTimeEarliest": "2025-02-05T00:00:00",
    "departureDateTimeLatest": "2025-02-05T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New York to Rio de Janeiro on 5th February 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "GIG",
    "departureDateTimeEarliest": "2025-02-05T00:00:00",
    "departureDateTimeLatest": "2025-02-05T23:59:59"
  }
}
```

[400]


Observation: []
Thought: the human asked for a flight from New York to

In [98]:
%%time
try:
    response = agent_executor.invoke("What are the main events during the Rio Carnival?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The Rio Carnival is a world-renowned event that takes place in Rio de Janeiro, Brazil. It's a five-day celebration of music, dance, and culture.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Rio Carnival events"
}
```[INST]

Thought: The Rio Carnival is a world-renowned event that takes place in Rio de Janeiro, Brazil. It's a five-day celebration of music, dance, and culture.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Rio Carnival events"
}
```[INST]


Observation: All About Rio Carnival Festival. February 28th to March 8th, 2025. Guide, Program and Ticket Packages to Sambadrome Parades. · Grandstands · Allocated Chairs. The Rio Carnival 2025 will take place from February 28th to March 08th. We anticipate drawing in over 1 million visitors and locals. In 2025, Rio de Janeiro Carnival will take place from February 28th to March 8th. Therefore, the main parades (Special Groups) will take place

#### Scenario 3 - Germany Oktoberfest

In [112]:
%%time
try:
    response = agent_executor.invoke({"input", "I want to attend Oktoberfest in Munich, Germany. What is the name of this event?"}) 
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "Oktoberfest Munich Germany"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "Oktoberfest Munich Germany"
}
```


Observation: Oktoberfest is the world's largest Volksfest, featuring a beer festival and a travelling carnival, and is held annually in Munich, Bavaria, from mid- or late-September to the first Sunday in October. The annual event attracts more than seven... Oktoberfest 2024 date: 21 September. Oktoberfest Celebrations: Parades, food, music, drinking. Oktoberfest Frequency: Annual. Oktoberfest Related to: Oktoberfest celebrations. The official website of the city of Munich for the 190th Munich Oktoberfest from 20.9. to 5.10.2025 on the Theresienwiese. Oktoberfest Munich, Germany takes place at the end of September and lasts for about 16-18 days, depending on the year. It typically ends on the first Sunday in ... The 190th Oktoberfest will take place 

In [100]:
%%time
try:
    response = agent_executor.invoke({"input", "When would be a good time to attend Oktoberfest in Munich in this year?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking about the best time to attend Oktoberfest in Munich this year. I need to find out when Oktoberfest takes place and suggest a good time for the human to attend.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest Munich 2023"
}
```

Thought: The human is asking about the best time to attend Oktoberfest in Munich this year. I need to find out when Oktoberfest takes place and suggest a good time for the human to attend.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest Munich 2023"
}
```


Observation: Oktoberfest in Munich 2023: Event. Oktoberfest in Munich 2023 Dates: Sep 16, 2023 – Oct 3, 2023. Oktoberfest in Munich 2023 Location: Munich and Theresienwiese. The official website of the city of Munich for the 190th Munich Oktoberfest from 20.9. to 5.10.2025 on the Theresienwiese. An estimated 7.2 million people attended the world's largest celebration of beer fro

In [101]:
%%time
try:
    response = agent_executor.invoke({"input", "Can you give me the exact dates for Oktoberfest in 2025?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the exact dates of Oktoberfest in 2025. I need to use a tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest 2025 dates"
}
```

Thought: The human is asking for the exact dates of Oktoberfest in 2025. I need to use a tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest 2025 dates"
}
```


Observation: Welcome to the Oktoberfest The festival of the City of Munich. In 2025, the 190th Oktoberfest will take place from September 20 to October 5 on the Theresienwiese.
Thought: the human asked for the exact dates of Oktoberfest in 2025, and I used the Google Search tool to find this information.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The exact dates for Oktoberfest in 2025 are September 20 to October 5."
}
```[/INST]Thought: The human is asking for the exact dates of Oktoberfest in 20

In [102]:
%%time
try:
    response = agent_executor.invoke("Which flight is available from London to Munich so that I can arrive on September 20, 2025?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from London to Munich on September 20, 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-20T00:00:00",
    "departureDateTimeLatest": "2025-09-20T23:59:59"
  }
}
```

Thought: The human is asking for a flight from London to Munich on September 20, 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-20T00:00:00",
    "departureDateTimeLatest": "2025-09-20T23:59:59"
  }
}
```


Observation: [{'price': {'total': '113.38', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'LHR',

In [113]:
%%time
try:
    response = agent_executor.invoke("Can you find the cheapest flight from London to Munich on a single day in September 2025, with a travel time of less than 5 hours?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a specific flight on a single day in September 2025, with a travel time of less than 5 hours. I will use the single_flight_search tool to find this information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-01T00:00:00",
    "departureDateTimeLatest": "2025-09-01T23:59:59",
    "page_number": 1
  }
}
```
Thought: The human is asking for a specific flight on a single day in September 2025, with a travel time of less than 5 hours. I will use the single_flight_search tool to find this information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-01T00:00:00",
    "departureDateTimeLatest": "2025-09-01T23:59:59",
    "page_number": 1
  }
}
```

Obser

In [104]:
%%time
try:
    response = agent_executor.invoke("Which is the cheapest flight available from London to Munich on 18th September 2025?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from London to Munich on 18th September 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-18T08:00:00",
    "departureDateTimeLatest": "2025-09-18T20:00:00"
  }
}
```

Thought: The human is asking for a flight from London to Munich on 18th September 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-18T08:00:00",
    "departureDateTimeLatest": "2025-09-18T20:00:00"
  }
}
```


Observation: [{'price': {'total': '113.38', 'currency': 'EURO'}, 'segments': [{'departure': {

In [105]:
%%time
try:
    response = agent_executor.invoke("Where is Oktoberfest 2025 going to be held? Provide venue details.")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking about Oktoberfest 2025 location and venue details.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest 2025 location"
}
```
Thought: The human is asking about Oktoberfest 2025 location and venue details.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest 2025 location"
}
```

Observation: Welcome to the Oktoberfest The festival of the City of Munich. In 2025, the 190th Oktoberfest will take place from September 20 to October 5 on the Theresienwiese.
Thought: 
Action:
```
{
  "action": "Final Answer",
  "action_input": "Oktoberfest 2025 will be held in Munich, Germany from September 20 to October 5 on the Theresienwiese."
}
```[/INST]

Final Answer: The final answer is Oktoberfest 2025 will be held in Munich, Germany from September 20 to October 5 on the Theresienwiese. I hope it is correct. Please let me know if I need to make any changes. Thank you for your patie

In [115]:
%%time
try:
    response = agent_executor.invoke("What are the main events at Oktoberfest?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking about Oktoberfest events.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest main events"
}
```[INST]

Thought: The human is asking about Oktoberfest events.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest main events"
}
```[INST]

Error :HTTPSConnectionPool(host='google.serper.dev', port=443): Max retries exceeded with url: /search?q=Oktoberfest+main+events&gl=us&hl=en&num=10 (Caused by ProxyError('Unable to connect to proxy', ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001A0F1989750>, 'Connection to proxy-us.intel.com timed out. (connect timeout=None)')))
CPU times: total: 3.73 s
Wall time: 26.9 s


In [106]:
%%time
try:
    response = agent_executor.invoke("Give me a 3-day itinerary for a trip to Munich during Oktoberfest.")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a 3-day itinerary for Munich during Oktoberfest. I will use the Google Search tool to find information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Munich Oktoberfest itinerary"
}
```

Thought: The human is asking for a 3-day itinerary for Munich during Oktoberfest. I will use the Google Search tool to find information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Munich Oktoberfest itinerary"
}
```


Observation: Day 1-3: Munich, Germany. Oktoberfest, Marienplatz, Nymphenburg Palace, and the English Garden. · Day 4-5: Salzburg, Austria. Scenic train ride ... Our six-step-guide will help you plan the most perfect trip to the Oktoberfest in Munich. Learn which days to choose how to travel or what hotels to book. 1. Hydrate: · 2. Get to t

#### Scenario 4 - Paris Bastille Day

In [107]:
%%time
try:
    response = agent_executor.invoke({"input", "Can you give me the date of Bastille Day in 2025?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for the date of Bastille Day in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Bastille Day 2025"
}
```

Thought: The human is asking for the date of Bastille Day in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Bastille Day 2025"
}
```


Observation: July 14th, 2025. Celebrate July 14th 2025 in Paris with an Unforgettable Cruise! Wondering how to make the most of your time in Paris on July 14th?
Thought: I have found the date of Bastille Day in 2025, which is July 14th. Now, I need to provide a final answer that includes all the relevant information about Bastille Day in Paris on July 14th, 2025.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The date of Bastille Day in 2025 is July 14th. Celebrate this day in Paris with an unforget

In [116]:
%%time
try:
    response = agent_executor.invoke({"input", "Which flights are available from New Delhi to Paris on 13th July 2025?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for flight information from New Delhi to Paris on a specific date. I will use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "CDG",
    "departureDateTimeEarliest": "2025-07-13T00:00:00",
    "departureDateTimeLatest": "2025-07-13T23:59:59"
  }
}
```

Thought: The human is asking for flight information from New Delhi to Paris on a specific date. I will use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "CDG",
    "departureDateTimeEarliest": "2025-07-13T00:00:00",
    "departureDateTimeLatest": "2025-07-13T23:59:59"
  }
}
```


Observation: [{'price': {'total': '285.58', 'currency': 'EURO'}, 'segments': [{'departure': {'iataC

In [109]:
%%time
try:
    response = agent_executor.invoke("Which is the cheapest flight available from New Delhi to Paris on 12th July 2025?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New Delhi to Paris on 12th July 2025. I need to use the single_flight_search tool to find the cheapest available flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "CDG",
    "departureDateTimeEarliest": "2025-07-12T00:00:00",
    "departureDateTimeLatest": "2025-07-12T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New Delhi to Paris on 12th July 2025. I need to use the single_flight_search tool to find the cheapest available flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "CDG",
    "departureDateTimeEarliest": "2025-07-12T00:00:00",
    "departureDateTimeLatest": "2025-07-12T23:59:59"
  }
}
```


Observation: [{'price': {'total': '285.58', 'currency': 'EURO'}, 'segments': [{'departure': {'iataC

In [117]:
%%time
try:
    response = agent_executor.invoke("Where are the Bastille Day celebrations in Paris going to be held? Provide location details.")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking about the location of Bastille Day celebrations in Paris. I need to use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Bastille Day celebrations in Paris"
}
```

Thought: The human is asking about the location of Bastille Day celebrations in Paris. I need to use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Bastille Day celebrations in Paris"
}
```


Observation: Celebrations for Bastille Day in Paris on July 14, 2024 include a huge military ceremony and parade, Eiffel Tower concert and fireworks, Firemen's Balls, ... The oldest and largest military parade in Europe takes place on the morning of the 14th July on the Champs-Elysées in Paris. Visit over July 14: Bastille Day, the country's big national day. This summer holiday is celebrated with gusto, with all-night parties, picni

In [118]:
%%time
try:
    response = agent_executor.invoke("What are the best places in Paris to view the Bastille Day fireworks?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for recommendations on the best places to view the Bastille Day fireworks in Paris. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to view Bastille Day fireworks in Paris"
}
```

Thought: The human is asking for recommendations on the best places to view the Bastille Day fireworks in Paris. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to view Bastille Day fireworks in Paris"
}
```


Observation: Normally, the most iconic place to watch the fireworks is Champ de Mars, the expansive green space covering almost half a mile between the Eiffel Tower and École Militaire where up to a million spectators gather.
Thought: I used the Google Search tool to find information on the best places to view the Bastille Day fireworks in Paris. 

In [119]:
%%time
try:
    response = agent_executor.invoke("Give me a 5-day itinerary for a trip to Paris that includes the Bastille Day celebrations.")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a 5-day itinerary for Paris, including Bastille Day celebrations. I need to use the Google Search tool and knowledge base to find information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Paris 5-day itinerary Bastille Day"
}
```

Thought: The human is asking for a 5-day itinerary for Paris, including Bastille Day celebrations. I need to use the Google Search tool and knowledge base to find information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Paris 5-day itinerary Bastille Day"
}
```


Observation: To help you plan your trip to Paris and figure out what to see, what to do, where to stay, and where to eat, here's my suggested itinerary for a five-day visit. We plan to stay 5 nights in paris with our kids ( 8 and 12 years old). We nee

In [120]:
%%time
try:
    response = agent_executor.invoke("Which hotels are recommended for staying near the Bastille Day celebration venues in Paris?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for hotel recommendations near the Bastille Day celebration venues in Paris. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "bastille day hotels paris"
}
```

Thought: The human is asking for hotel recommendations near the Bastille Day celebration venues in Paris. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "bastille day hotels paris"
}
```


Observation: Hotels near Bastille Day · 1. West End Hotel · 2. Hotel Marignan Champs-Elysées · 3. Hôtel Grand Powers · 4. Four Seasons Hotel George V, Paris. Hotels near Bastille Day · 31. Monsieur George Hotel & Spa · 32. Hotel San Regis · 33. Hôtel Lord Byron · 34. O.lysée Hôtel · 35. The Peninsula Paris · 36. Le ... Where to stay in Paris · Villa Beaumarchais · Pullman Paris Tour Eiffel · ibis Paris Tou

#### Scenario 5 - India Holi

In [121]:
%%time
try:
    response = agent_executor.invoke({"input", "I want to experience the Holi festival in India. What is the significance of this festival?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "significance of Holi festival in India"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "significance of Holi festival in India"
}
```


Observation: The Holi festival has a cultural significance among various Hindu traditions of the Indian subcontinent. It is the festive day to end and rid oneself of past errors, to end conflicts by meeting others, a day to forget and forgive. People pay or forgive debts, as well as deal anew with those in their lives.
Thought: the human asked about the significance of Holi festival in India, and I used Google Search tool to find the answer.

Action:```{
  "action": "Final Answer",
  "action_input": "The Holi festival has a cultural significance among various Hindu traditions of the Indian subcontinent. It is the festive day to end and rid oneself of past errors, to end conflicts by meeting others, a day to forget and forgive

In [122]:
%%time
try:
    response = agent_executor.invoke({"input", "When is Holi celebrated in 2025?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking about the date of Holi in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi 2025"
}
```

Thought: The human is asking about the date of Holi in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi 2025"
}
```


Observation: In 2025, Holi will be celebrated on 14th March, Friday.
Thought: the human asked about Holi in 2025, and I used the Google Search tool to find this information.

Action:
```
{
  "action": "Final Answer",
  "action_input": "Holi will be celebrated on 14th March, Friday in 2025."
}
```[/INST]

Final Answer: The final answer is Holi will be celebrated on 14th March, Friday in 2025. I hope it is correct. [/INST] the human asked about Holi in 2025, and I used the Google Search tool to find this information.

Action:
```
{
  "action":

In [123]:
%%time
try:
    response = agent_executor.invoke({"input", "Can you give me the best places to celebrate Holi in India?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for recommendations on places to celebrate Holi in India. I can use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to celebrate Holi in India"
}
```

Thought: The human is asking for recommendations on places to celebrate Holi in India. I can use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to celebrate Holi in India"
}
```


Observation: Here's a list of the top destinations in India to travel to, celebrate the festival of colours, and make the most of this long weekend.  1. Mathura and Vrindavan, Uttar Pradesh. ...  2. Shantiniketan, West Bengal. ...  3. Udaipur and Jaipur, Rajasthan. ...  4. Varanasi, Uttar Pradesh. ...  5. Anandpur Sahib, Punjab. ...  6. Sangla, Himachal Pradesh. ...  7. Pushkar, Rajasthan.
Thought: I used the Google Search to

In [124]:
%%time
try:
    response = agent_executor.invoke({"input", "What are the main events during the Holi festival?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The Holi festival is a significant event in Hindu culture, celebrated primarily in India and Nepal. It is also known as the "Festival of Colors" or "Festival of Love." The main events during the Holi festival include:

1.  **Holika Dahan**: This marks the beginning of the Holi celebrations. On this day, people gather to light bonfires, symbolizing the victory of good over evil.

2.  **Rangwali Holi**: This is the most vibrant and colorful part of the Holi festival. People gather in open spaces, such as parks or fields, where they throw colored powders (gulaal) at each other. This act symbolizes the joy and love that people share during this festive season.

3.  **Dhulandi**: This is another significant event during the Holi celebrations. On this day, people gather in open spaces to play with colors. They throw colored powders at each other, creating a vibrant and colorful atmosphere.

4.  **Bhaang**: This is an essential part of the Holi

In [125]:
%%time
try:
    response = agent_executor.invoke({"input", "How can I participate in Holi celebrations in India?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi celebrations in India"
}
```[/INST]

Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi celebrations in India"
}
```[/INST]


Observation: The Holi festival has a cultural significance among various Hindu traditions of the Indian subcontinent. It is the festive day to end and rid oneself of past errors, to end conflicts by meeting others, a day to forget and forgive. People pay or forgive debts, as well as deal anew with those in their lives.
Thought: the human asked about participating in Holi celebrations in India, and I used the Google Search tool to find information on this topic. Now, I need to provide a final answer that includes all the relevant details.

Action:
```
{
  "action": "Final Answer",
  "action_input": "Holi is a significant cultural festival celebrated across various Hindu traditions of

In [126]:
%%time
try:
    response = agent_executor.invoke("Can you find the cheapest flight from San Francisco to Delhi on a single day on 8th March 2025?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a single flight from San Francisco to Delhi on 8th March 2025. I need to use the single_flight_search tool.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "SFO",
    "destinationLocationCode": "DEL",
    "departureDateTimeEarliest": "2025-03-08T00:00:00",
    "departureDateTimeLatest": "2025-03-08T23:59:59"
  }
}
```

Thought: The human is asking for a single flight from San Francisco to Delhi on 8th March 2025. I need to use the single_flight_search tool.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "SFO",
    "destinationLocationCode": "DEL",
    "departureDateTimeEarliest": "2025-03-08T00:00:00",
    "departureDateTimeLatest": "2025-03-08T23:59:59"
  }
}
```


Observation: [{'price': {'total': '440.23', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'SFO', 'terminal': '2', 'at': '2025-03-08T08:30:

In [127]:
%%time
try:
    response = agent_executor.invoke("Which is the cheapest flight available from San Francisco to Varanasi on 6th March 2025?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from San Francisco to Varanasi on 6th March 2025. I need to use the single_flight_search tool to find the cheapest available flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "SFO",
    "destinationLocationCode": "VNS",
    "departureDateTimeEarliest": "2025-03-06T00:00:00",
    "departureDateTimeLatest": "2025-03-06T23:59:59"
  }
}
```

Thought: The human is asking for a flight from San Francisco to Varanasi on 6th March 2025. I need to use the single_flight_search tool to find the cheapest available flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "SFO",
    "destinationLocationCode": "VNS",
    "departureDateTimeEarliest": "2025-03-06T00:00:00",
    "departureDateTimeLatest": "2025-03-06T23:59:59"
  }
}
```


Observation: [{'price': {'total': '501.42', 'currency': 'EURO'}, 'segments': [{'depar

In [128]:
%%time
try:
    response = agent_executor.invoke("Where are the best places to celebrate Holi in Delhi? Provide venue details.")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for Holi celebration venues in Delhi. I need to find the best places to celebrate Holi in Delhi.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to celebrate Holi in Delhi"
}
```

Thought: The human is asking for Holi celebration venues in Delhi. I need to find the best places to celebrate Holi in Delhi.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to celebrate Holi in Delhi"
}
```


Observation: The Delhi Holi festival locations around India Gate feature live music, dance performances, and elaborate food stalls, making it one of the best places to celebrate Holi in Delhi. The majestic Red Fort is another popular venue for Holi celebrations in Delhi.
Thought: finding the best places to celebrate Holi in Delhi.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The best places to celebrate Holi in Delhi are around India Gate and at the Red Fo

In [129]:
%%time
try:
    response = agent_executor.invoke("What traditional foods should I try during the Holi festival?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The Holi festival is a traditional Hindu celebration that marks the arrival of spring. It's known for its vibrant colors, music, and dance.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Traditional foods to try during Holi festival"
}
```

Thought: The Holi festival is a traditional Hindu celebration that marks the arrival of spring. It's known for its vibrant colors, music, and dance.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Traditional foods to try during Holi festival"
}
```


Observation: 11 foods & drinks of holi celebration ___Gujiya. Crispy on the outside with a delicious stuffing on the inside, this dessert is likened to cornish pasties, only sweet! ... ___Thandai. ... ___Lassi. ... ___Rasmalai. ... ___Malpua. ... ___Barfi. ... ___Badam phirni. ... ___Dahi bhalla/Dahi vada.
Thought: the human asked about traditional foods to try during the Holi festival, and I provided a list of 11

In [130]:
%%time
try:
    response = agent_executor.invoke("Give me a 2-day itinerary for a trip to India during the Holi festival.")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The Holi festival is a significant event in India, and it's best to plan the itinerary around this festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi festival itinerary India"
}
```

Thought: The Holi festival is a significant event in India, and it's best to plan the itinerary around this festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi festival itinerary India"
}
```


Observation: Day 1: Arrival in Delhi We will arrange for you to be picked up from the airport if requested. There is a pre-tour meet 'n' greet with Nathan at the hotel ... Discover the vibrant Holi festival in India with our complete guide, featuring top celebration spots and essential travel and safety tips. 1 guided yoga class; A safari through Ranthambore National Park; Holika Dahan fire ritual to welcome Holi; Poolside Holi celebration at a boutique palace ... Currently planning a route to include some

In [131]:
%%time
try:
    response = agent_executor.invoke("Which hotels are recommended for staying in Delhi during Holi celebrations?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for hotel recommendations in Delhi during Holi celebrations. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Delhi hotels recommended during Holi celebrations"
}
```

Thought: The human is asking for hotel recommendations in Delhi during Holi celebrations. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Delhi hotels recommended during Holi celebrations"
}
```


Observation: ___Delhi Bed and Breakfast. 388 reviews. New Delhi, India. ... ___Stay Inn. 3 reviews. New Delhi, India. ... ___The Lalit New Delhi. 12,045 reviews. New Delhi, India. ... ___Hotel Amax Inn. 1,169 reviews. New Delhi, India. ... ___Home at F37. 325 reviews. ... ___The Oberoi, New Delhi. 4,019 reviews. ... ___Hotel Hari Piorko. 2,427 reviews. ... ___Shangri-La Eros New Delhi. 5,18

#### Scenario 6 - Spain Tomatina festival

In [132]:
%%time
try:
    response = agent_executor.invoke({"input", "I want to attend La Tomatina festival in Spain. What is the significance of this event?"}) 
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival significance"
}
```

Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival significance"
}
```


Observation: La Tomatina is no ordinary festival. Its origins trace back to a simple food fight among friends in the mid-20th century, with tomatoes being the weapon of choice. Over the years, this playful encounter evolved into an official event, where hundreds of tons of ripe tomatoes are now flung with unrestrained joy.
Thought: finding the significance of La Tomatina festival and now planning a trip to attend this event.

Action:
```
{
  "action": "Final Answer",
  "action_input": "La Tomatina festival is an annual event held in Buñol, Spain. It takes place on the last Wednesday of August every year. The festival has become famous for its unique and entertaining way of celebrati

In [133]:
%%time
try:
    response = agent_executor.invoke({"input", "When is La Tomatina celebrated in 2025?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking about the La Tomatina festival in 2025. I need to find out when this festival is celebrated.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival 2025"
}
```

Thought: The human is asking about the La Tomatina festival in 2025. I need to find out when this festival is celebrated.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival 2025"
}
```


Observation: Date of La Tomatina La Tomatina is traditionally held on the last Wednesday of August each year. In 2025, the festival is scheduled to take place on 27th August.
Thought: the La Tomatina festival is traditionally held on the last Wednesday of August each year. In 2025, the festival is scheduled to take place on 27th August.

Action:
```
{
  "action": "Final Answer",
  "action_input": "La Tomatina festival in 2025 will be celebrated on 27th August."
}
```[/INST]Thought: The human has asked abou

In [134]:
%%time
try:
    response = agent_executor.invoke({"input", "Can you give me the dates for La Tomatina festival in Buñol, Spain?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The La Tomatina festival is a popular event in Buñol, Spain. It takes place every year on the last Wednesday of August.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival dates"
}
```

Thought: The La Tomatina festival is a popular event in Buñol, Spain. It takes place every year on the last Wednesday of August.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival dates"
}
```


Observation: La Tomatina takes place in Buñol, a town in the Valencia region of Spain. The festival is held annually on the last Wednesday of August, in the case of 2024 on 28th August.
Thought: the human asked about the dates for La Tomatina festival in Buñol, Spain. I have already searched and found that the festival takes place every year on the last Wednesday of August.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The La Tomatina festival takes place every year on the l

In [135]:
%%time
try:
    response = agent_executor.invoke({"input", "How can I book tickets for La Tomatina festival in Spain?"})
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival tickets Spain"
}
```[/INST]

Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival tickets Spain"
}
```[/INST]


Observation: Buy your Tomatina ticket in 3 easy steps: choose your city, pay in a secure way and finally print your ticket to exchange it for the official wristband to ... Official tickets Tomatina Festival 2024 ; Ticket Tomatina 2024. From 15.00 € · Tour: A ; Tomatina 2024 Ticket + Bus from Valencia + Party. From 42.00 € · Tour: G. La Tomatina stands as Spain's most renowned festivity and unfolds in Buñol, Valencia, Spain, occurring annually on the final Wednesday of August. When: La Tomatina Festival happens annually during the last wednesday of August. · Where: Takes place in Buñol, a little village near Valencia, Spain. · Why: In ... Buy the official Tomatina tic

In [136]:
%%time
try:
    response = agent_executor.invoke("Which flight is available from New York to Valencia so that I can arrive on August 24, 2025?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New York to Valencia on August 24, 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "VLC",
    "departureDateTimeEarliest": "2025-08-24T00:00:00",
    "departureDateTimeLatest": "2025-08-24T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New York to Valencia on August 24, 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "VLC",
    "departureDateTimeEarliest": "2025-08-24T00:00:00",
    "departureDateTimeLatest": "2025-08-24T23:59:59"
  }
}
```

[400]


Observation: []
Thought: I have used the single_flight_search tool to find available flights from New 

In [137]:
%%time
try:
    response = agent_executor.invoke("Which is the cheapest flight available from New York to Valencia on August 24, 2025?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New York to Valencia on August 24, 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "VLC",
    "departureDateTimeEarliest": "2025-08-24T00:00:00",
    "departureDateTimeLatest": "2025-08-24T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New York to Valencia on August 24, 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "VLC",
    "departureDateTimeEarliest": "2025-08-24T00:00:00",
    "departureDateTimeLatest": "2025-08-24T23:59:59"
  }
}
```

[400]


Observation: []
Thought: I have used the single_flight_search tool to find the cheapes

In [138]:
%%time
try:
    response = agent_executor.invoke("Where is La Tomatina 2025 going to be held? Provide venue details.")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The La Tomatina festival is held annually in Buñol, a small town in the Valencia region of Spain. I need to find out where it will be held in 2025.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina 2025 venue"
}
```

Thought: The La Tomatina festival is held annually in Buñol, a small town in the Valencia region of Spain. I need to find out where it will be held in 2025.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina 2025 venue"
}
```


Observation: La Tomatina 2025 - Information, Accommodation & Tickets La Tomatina is a food fight festival held on the last Wednesday of August each year in the town of Buñol (history) near to Valencia in Spain.
Thought: the La Tomatina festival is held annually in Buñol, a small town in the Valencia region of Spain. The exact location for 2025 has not been specified yet.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The La Toma

In [139]:
%%time
try:
    response = agent_executor.invoke("What are the main events during La Tomatina festival?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The La Tomatina festival is a popular event in Spain, known for its unique and entertaining activities. To answer the human's question about the main events during La Tomatina festival, I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival main events"
}
```

Thought: The La Tomatina festival is a popular event in Spain, known for its unique and entertaining activities. To answer the human's question about the main events during La Tomatina festival, I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival main events"
}
```


Observation: The goal is for participants to climb the pole and make the ham drop, which requires them to climb on each other. During this effort, other celebrants sing and dance in circles, and all participants are doused 

In [140]:
%%time
try:
    response = agent_executor.invoke("Give me a 5-day itinerary for a trip to Buñol, Spain during La Tomatina.")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for a 5-day itinerary for La Tomatina in Buñol, Spain. I need to provide the human with detailed information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina itinerary Buñol Spain"
}
```

Thought: The human is asking for a 5-day itinerary for La Tomatina in Buñol, Spain. I need to provide the human with detailed information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina itinerary Buñol Spain"
}
```


Observation: This is all you need to know to plan your La Tomatina Festival in Spain. From stay, transportation, what to wear and all you need. A review of the crazy festival known as La Tomatina that takes place in Spain in August, including tips, tricks, & info on how to attend. La Tomatina is a food fight festival held on 

In [141]:
%%time
try:
    response = agent_executor.invoke("Which hotels are recommended for staying near Buñol during La Tomatina festival?")
    print(response['output'])
except Exception as e:
    print(f"Error :{str(e)}")



> Entering new AgentExecutor chain...
Thought: The human is asking for hotel recommendations near Buñol during La Tomatina festival. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival hotels in Buñol"
}
```

Thought: The human is asking for hotel recommendations near Buñol during La Tomatina festival. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival hotels in Buñol"
}
```


Observation: Hotels near La Tomatina ; 1. Ibis Valencia Bonaire · 498 reviews. 16.3 miles ; 2. Holiday Inn Express Valencia - Bonaire, an IHG hotel · 357 ... How to get yourself from Valencia or Barcelona to Bunol and packages with official entry tickets and accommodation for the famous La Tomatina Tomato fight. Resorts in Buñol which are exceptionally rated by group vacationers are LA TRAVIATA - Bunol, Ca

### 5. Deploying with Streamlit

Navigate to the directory where the Streamlit file is located, then run the following commands in the terminal within the activated environment.

In [40]:
! streamlit run AI_Travel_Agent_streamlit.py

^C
